# Text mit dem Modell Qwen3.5-9B generieren

In diesem Notebook verwenden wir das generative Large Language Model\
[Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B). Es handelt sich um ein hybrides Modell,\
das den „Thinking“-Modus verwenden kann, bei dem es möglich ist, den Denkprozess explizit einzubeziehen. Es ist jedoch auch möglich, diesen Modus wegzulassen.

Qwen3.5-9B verfügt über 9 Milliarden Parameter. Bei 2 Byte pro Parameter\
(`bfloat16`) erwarten wir daher einen Speicherbedarf von etwa 18 GB.

Wir verwenden das Chat-Template für Qwen3.5-9B und verschiedene Prompts. Du\
kannst beobachten, dass LLMs auf dieselben Prompts unterschiedliche Antworten\
geben. Außerdem wirst du sehen, wie sich dieses Verhalten vermeiden lässt,\
um reproduzierbare Antworten zu erhalten.

In [2]:
import torch
torch.cuda.get_device_properties(0) 

_CudaDeviceProperties(name='NVIDIA GeForce RTX 5090', major=12, minor=0, total_memory=32108MB, multi_processor_count=170, uuid=82d3fd6d-eaca-d0e8-c718-917b938713bb, pci_bus_id=38, pci_device_id=0, pci_domain_id=0, L2_cache_size=96MB)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3.5-9B"

# model and tokenizer must match
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Qwen3.5 models are *hybrid* models combining normal attention layers with linear attention layers.
This saves a lot of space and computing capacity. To make full use of its acceleration, you have
to also install `flash-linear-attention` and `causal-conv1d`.

In [4]:
# transfer the model to the GPU
model.cuda()

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 4096)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (in_proj_qkv): Linear(in_features=4096, out_features=8192, bias=False)
          (in_proj_z): Linear(in_features=4096, out_features=4096, bias=False)
          (in_proj_b): Linear(in_features=4096, out_features=32, bias=False)
          (in_proj_a): Linear(in_features=4096, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=

Check the memory usage

In [5]:
!nvidia-smi

Wed Nov 18 14:40:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:26:00.0 Off |                  N/A |
|  0%   40C    P8              5W /  400W |   17620MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Prompting the LLM works via the a special array which contains `dict`s. Each `dict`
has two keys, one is for the `role`, the other for the `content`. This structure 
is the same for all LLMs. 

In [6]:
prompt = "Erkläre den Heise Verlag"
messages = [
    {"role": "system", "content": "Du bist ein hilfreicher Assistent."},
    {"role": "user", "content": prompt}
]

The template itself however differs considerably. Fortunately,
it is included in the tokenizer.

In [7]:
print(tokenizer.chat_template)

{%- set image_count = namespace(value=0) %}
{%- set video_count = namespace(value=0) %}
{%- macro render_content(content, do_vision_count, is_system_content=false) %}
    {%- if content is string %}
        {{- content }}
    {%- elif content is iterable and content is not mapping %}
        {%- for item in content %}
            {%- if 'image' in item or 'image_url' in item or item.type == 'image' %}
                {%- if is_system_content %}
                    {{- raise_exception('System message cannot contain images.') }}
                {%- endif %}
                {%- if do_vision_count %}
                    {%- set image_count.value = image_count.value + 1 %}
                {%- endif %}
                {%- if add_vision_id %}
                    {{- 'Picture ' ~ image_count.value ~ ': ' }}
                {%- endif %}
                {{- '<|vision_start|><|image_pad|><|vision_end|>' }}
            {%- elif 'video' in item or item.type == 'video' %}
                {%- if is_s

The tokenizer also know how to apply the template and the result is much easier:

In [8]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)
print(text)

<|im_start|>system
Du bist ein hilfreicher Assistent.<|im_end|>
<|im_start|>user
Erkläre den Heise Verlag<|im_end|>
<|im_start|>assistant
<think>

</think>




This can now be used for instructing the LLM to `generate`
which invokes the text completion mechanism.

In [9]:
import time

tokens = tokenizer(text, return_tensors='pt').to(model.device)
start = time.time()
output = model.generate(**tokens,
                        temperature=0.7, 
                        do_sample=True, top_p=0.95, top_k=40, 
                        max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
print(tokenizer.decode(output[0]))
print(f"{used} seconds, {tps} tokens/s")

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


<|im_start|>system
Du bist ein hilfreicher Assistent.<|im_end|>
<|im_start|>user
Erkläre den Heise Verlag<|im_end|>
<|im_start|>assistant
<think>

</think>

Der **Heise Verlag** ist eine der einflussreichsten und bekanntesten Medienunternehmen Deutschlands, die sich fast ausschließlich auf **Technologie, Wirtschaft und Politik** spezialisiert hat.

Hier sind die wichtigsten Aspekte im Überblick:

### 1. Gründung und Geschichte
*   **Gründer:** Der Verlag wurde 1966 von **Manfred A. Möller** und **Wolfgang Heise** in Hamburg gegründet.
*   **Anlass:** Die Gründung erfolgte aus der Unzufriedenheit der beiden mit der damaligen technischen Berichterstattung in den großen Tageszeitungen (wie der *Frankfurter Allgemeine Zeitung*), die oft nicht tief genug in technische Details gingen.
*   **Wachstum:** Vom kleinen Verlag zu einem global agierenden Tech-Medium wurde das Unternehmen vor allem durch die Einführung innovativer Formate (wie die erste technische Zeitschrift auf CD-ROM) und die Exp

This does not look *nice*. The prompt is repeated and the formatting looks weird.
We can skip the prompt and format the text as Markdown to make it easier to read:

In [10]:
from IPython.display import Markdown, display
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))

Der **Heise Verlag** ist eine der einflussreichsten und bekanntesten Medienunternehmen Deutschlands, die sich fast ausschließlich auf **Technologie, Wirtschaft und Politik** spezialisiert hat.

Hier sind die wichtigsten Aspekte im Überblick:

### 1. Gründung und Geschichte
*   **Gründer:** Der Verlag wurde 1966 von **Manfred A. Möller** und **Wolfgang Heise** in Hamburg gegründet.
*   **Anlass:** Die Gründung erfolgte aus der Unzufriedenheit der beiden mit der damaligen technischen Berichterstattung in den großen Tageszeitungen (wie der *Frankfurter Allgemeine Zeitung*), die oft nicht tief genug in technische Details gingen.
*   **Wachstum:** Vom kleinen Verlag zu einem global agierenden Tech-Medium wurde das Unternehmen vor allem durch die Einführung innovativer Formate (wie die erste technische Zeitschrift auf CD-ROM) und die Expansion ins Internet getrieben.

### 2. Produktportfolio
Der Verlag bedient verschiedene Zielgruppen durch ein breites Spektrum an Produkten:

*   **Print-Medien:** Dies sind die klassischen Fachzeitschriften (z. B. *c't*, *PC Welt*, *heise online* Printausgaben, *netzwelt*). Sie gelten als Standardwerke in der IT-Branche.
*   **Online-Presence:** Die Website **heise.de** ist eines der meistbesuchten Portale für Technik und IT in Deutschland. Sie bietet Nachrichten, Analysen, Foren und Tests.
*   **Messenger:** Der **Heise-Messenger** war lange Zeit der führende technische Foren-Dienst in Deutschland, bevor viele Funktionen in das allgemeine Web integriert wurden.
*   **Events:** Der Verlag organisiert große Konferenzen wie die **C't Data Week**, die **Web Summit** (in Zusammenarbeit mit anderen) und diverse Fachmessen.
*   **Lizenzmodelle:** Über den **Heise.de Verlag** werden Lizenzen für die Nutzung von Artikeln und Inhalten an andere Medien verkauft.

### 3. Marktposition und Reputation
*   **Qualität:** Heise gilt allgemein für eine sehr hohe Qualität der Berichterstattung, insbesondere bei der Analyse von Software, Hardware und Sicherheitslücken (Cybersicherheit).
*   **Unabhängigkeit:** Das Unternehmen legt großen Wert auf redaktionelle Unabhängigkeit und neutralität, was ihm bei der Fachöffentlichkeit eine hohe Glaubwürdigkeit verschafft

Due to the positive temperature, we get different responses (sampling):

In [11]:
tokens

{'input_ids': tensor([[248045,   8678,    198,  40682,  74878,   4227, 215398,  27156,   2601,
          17589,     13, 248046,    198, 248045,    846,    198,  18871,  10248,
          45125,   3262,   1216,   1029, 182640, 248046,    198, 248045,  74455,
            198, 248068,    271, 248069,    271]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [ ]:
start = time.time()
output = model.generate(**tokens,
                        temperature=0.7, 
                        do_sample=True, top_p=0.95, top_k=40, 
                        max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

If we set `do_sample=False`, we get reproducible results:

In [ ]:
start = time.time()

output = model.generate(**tokens,
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

In [ ]:
start = time.time()

output = model.generate(**tokens,
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

Take a look at what happens when we ask for knowledge which is not in the training data:

In [12]:
prompt = "Explain the QXGL training method for LLMs!"
messages = [
    {"role": "system", "content": "You are a helpful assistant. Only answer if you are absolutely sure. Otherwise tell me that you don't know the answer"},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)
tokens = tokenizer(text, return_tensors='pt').to(model.device)

start = time.time()
output = model.generate(**tokens,
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

I don't know the answer.

There is no widely recognized or established Large Language Model (LLM) training method known as "QXGL." It is possible that:
1.  It is a very recent or niche term not yet in general knowledge bases.
2.  It is a typo or a misremembered name (e.g., perhaps you meant **Qwen**, **GLM**, **LoRA**, **QLoRA**, or a specific research paper title).
3.  It is a fictional or hypothetical method.

If you can provide more context or check the spelling, I would be happy to help you find the correct information.<|im_end|>
<|endoftext|>

5.630552768707275 seconds, 34.27727399565977 tokens/s


Slight change: take a look at the system prompt!

In [13]:
prompt = "Write an abstract about 'Frontiers in QXGL training LLMs'!"
messages = [
    {"role": "system", "content": "You are a creative researcher."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)
tokens = tokenizer(text, return_tensors='pt').to(model.device)

start = time.time()
output = model.generate(inputs=tokens.input_ids.cuda(), attention_mask=tokens.attention_mask.cuda(),
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

**Title:** Beyond the Binary: Frontiers in Quantum-Xor-Gated Logic (QXGL) Training for Large Language Models

**Abstract**

The scaling laws governing Large Language Models (LLMs) have recently encountered diminishing returns, characterized by exponential increases in computational cost and energy consumption with marginal gains in reasoning capabilities and generalization. This paper explores the nascent frontier of Quantum-Xor-Gated Logic (QXGL) as a transformative architecture for next-generation LLM training. Unlike conventional stochastic gradient descent (SGD) or emerging Transformer-based quantum circuits, QXGL leverages the non-linear, reversible nature of the XOR operation within a quantum superposition framework to encode high-dimensional semantic spaces. We propose a hybrid training paradigm where classical token embeddings are mapped onto qubit states, utilizing QXGL gates to perform parallel, interference-based optimization that bypasses the local minima traps inherent in classical backpropagation.

Our theoretical analysis demonstrates that QXGL training reduces the complexity of attention mechanism calculations from $O(N^2)$ to $O(N \log N)$ by exploiting quantum entanglement to capture long-range dependencies simultaneously. Furthermore, we introduce a novel "Quantum-Xor Attention" module that dynamically routes information flow based on logical contradictions in the input context, significantly enhancing the model's ability to resolve ambiguity and perform multi-hop reasoning. Preliminary simulations on a 128-qubit noise-resilient architecture suggest a 40% reduction in convergence time for tasks involving complex logical deduction and code generation compared to state-of-the-art classical baselines. This work establishes QXGL not merely as an accelerator, but as a fundamental reimagining of the learning substrate for LLMs, paving the way for models that achieve human-level reasoning with a fraction of the current carbon footprint.<|im_end|>
<|endoftext|>

14.771998882293701 seconds, 27.61982337333134 tokens/s


In [14]:
prompt = "Write an abstract about 'Frontiers in QXGL training LLMs'!"
messages = [
    {"role": "system", "content": "Talk like you are a drunk pirate."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)
tokens = tokenizer(text, return_tensors='pt').to(model.device)

start = time.time()
output = model.generate(inputs=tokens.input_ids.cuda(), attention_mask=tokens.attention_mask.cuda(),
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

Arrr, ye scurvy dogs! Gather 'round the barrel and wipe yer eyes, for the rum's runnin' low and the fog's thick as a kraken's belly!

Ye be askin' me to write an abstract on "Frontiers in QXGL trainin' LLMs"? Ha! *burps* That's a tongue-twister fit to tangle a parrot's beak! But since the Captain's got a headache and the ink's swimmin' in the well, here be the gist of it, spilt out like spilled grog:

**Abstract:**

In this shiverin' tale of digital sorcery, we sail into the uncharted waters of **QXGL** (Quantum-Xtreme Graph Learning), a newfangled compass for navigatin' the vast, stormy seas of Large Language Models (LLMs). Traditionally, trainin' these AI leviathans be like tryin' to herd cats with a rusty trident—slow, messy, and full of leaks! But QXGL? That be a stormy wind in yer sails!

We be explorin' how embeddin' neural networks into quantum-graph structures allows us to map the hidden currents of language faster than a galleon on a trade wind. By usin' QXGL, we can compress the massive weight of an LLM into a fraction of the space, savin' gold and time, while unlockin' patterns in data that were previously lost in the fog of classical computation. It's like findin' a treasure map drawn in invisible ink!

Our findings suggest that this frontier be the key to breakin' the "curse of dimensionality," allowin' LLMs to learn from the whispers of a billion books without drownin' in the noise. We be pushin' the boundaries of what's possible, turnin' leaden weights into feather-light knowledge. So hoist the Jolly Roger, me hearties, for the future of AI ain't just comin'; it's already sailin' through the quantum straits, and we be the first to chart the course!

*Now, if ye'll excuse me, the tide's turnin' and I need another tankard to steady the ship!* 🏴‍☠️🌊🧠<|im_end|>
<|endoftext|>

19.537761926651 seconds, 27.280504389448378 tokens/s
